# NETWORK INTRUSION DETECTION - MODEL TRAINING

## Model Training for Network Intrusion Detection

#### 1.1 Import Data and Required Packages
##### Importing Pandas, Numpy, Matplotlib, Seaborn and Warnings Library.


In [4]:
!pip install xgboost catboost lightgbm scikit-optimize seaborn

# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Modelling
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB

print("✅ All packages imported successfully!")

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
✅ All packages imported successfully!


#### Import the CSV Data as Pandas DataFrame


In [6]:
# Load training and testing datasets
df_train = pd.read_csv('data/UNSW_NB15_training-set.csv')
df_test = pd.read_csv('data/UNSW_NB15_testing-set.csv')

print("Training set shape:", df_train.shape)
print("Testing set shape:", df_test.shape)

Training set shape: (175341, 45)
Testing set shape: (82332, 45)


#### Merge datasets for comprehensive training

In [7]:
df = pd.concat([df_train, df_test], ignore_index=True)
print("Combined dataset shape:", df.shape)

Combined dataset shape: (257673, 45)


#### Show Top 5 Records

In [8]:
df.head()

,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,...,1,1,0,0,0,1,1,0,Normal,0
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,...,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,...,1,3,0,0,0,2,6,0,Normal,0
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,...,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,...,1,40,0,0,0,2,39,0,Normal,0


#### Data Preprocessing


In [9]:
# Remove unnecessary columns
df.drop(['id'], axis=1, inplace=True, errors='ignore')

print("Categories in 'proto' variable:     ", end=" ")
print(df['proto'].unique())

print("Categories in 'service' variable:  ", end=" ")
print(df['service'].unique())

print("Categories in 'state' variable:     ", end=" ")
print(df['state'].unique())

print("Categories in 'attack_cat' variable:", end=" ")
print(df['attack_cat'].unique())

print("Categories in 'label' variable:     ", end=" ")
print(df['label'].unique())

Categories in 'proto' variable:      ['tcp' 'udp' 'arp' 'ospf' 'icmp' 'igmp' 'rtp' 'ddp' 'ipv6-frag' 'cftp'
 'wsn' 'pvp' 'wb-expak' 'mtp' 'pri-enc' 'sat-mon' 'cphb' 'sun-nd' 'iso-ip'
 'xtp' 'il' 'unas' 'mfe-nsp' '3pc' 'ipv6-route' 'idrp' 'bna' 'swipe'
 'kryptolan' 'cpnx' 'rsvp' 'wb-mon' 'vmtp' 'ib' 'dgp' 'eigrp' 'ax.25'
 'gmtp' 'pnni' 'sep' 'pgm' 'idpr-cmtp' 'zero' 'rvd' 'mobile' 'narp' 'fc'
 'pipe' 'ipcomp' 'ipv6-no' 'sat-expak' 'ipv6-opts' 'snp' 'ipcv'
 'br-sat-mon' 'ttp' 'tcf' 'nsfnet-igp' 'sprite-rpc' 'aes-sp3-d' 'sccopmce'
 'sctp' 'qnx' 'scps' 'etherip' 'aris' 'pim' 'compaq-peer' 'vrrp' 'iatp'
 'stp' 'l2tp' 'srp' 'sm' 'isis' 'smp' 'fire' 'ptp' 'crtp' 'sps'
 'merit-inp' 'idpr' 'skip' 'any' 'larp' 'ipip' 'micp' 'encap' 'ifmp'
 'tp++' 'a/n' 'ipv6' 'i-nlsp' 'ipx-n-ip' 'sdrp' 'tlsp' 'gre' 'mhrp' 'ddx'
 'ippc' 'visa' 'secure-vmtp' 'uti' 'vines' 'crudp' 'iplt' 'ggp' 'ip'
 'ipnip' 'st2' 'argus' 'bbn-rcc' 'egp' 'emcon' 'igp' 'nvp' 'pup' 'xnet'
 'chaos' 'mux' 'dcn' 'hmp' 'prm' 'trunk-1' 'xn

#### Encoding Categorical Variables


In [10]:
# Service encoding
service_mapping = {
    '-': 0, 'ftp': 1, 'smtp': 2, 'snmp': 3, 'http': 4, 'ftp-data': 5,
    'dns': 6, 'ssh': 7, 'radius': 8, 'pop3': 9, 'dhcp': 10, 'ssl': 11, 'irc': 12
}
df['service'] = df['service'].map(service_mapping).fillna(0).astype(int)

# State encoding
state_mapping = {
    'FIN': 1, 'INT': 2, 'CON': 3, 'ECO': 4, 'REQ': 5, 'RST': 6,
    'PAR': 7, 'URN': 8, 'no': 9, 'ACC': 10, 'CLO': 11
}
df['state'] = df['state'].map(state_mapping).fillna(0).astype(int)

# Protocol encoding using LabelEncoder
le_proto = LabelEncoder()
df['proto'] = le_proto.fit_transform(df['proto'])

# Attack category encoding
attack_cat_mapping = {
    'Normal': 0, 'Backdoor': 1, 'Analysis': 2, 'Fuzzers': 3, 'Shellcode': 4,
    'Reconnaissance': 5, 'Exploits': 6, 'DoS': 7, 'Worms': 8, 'Generic': 9
}
df['attack_cat'] = df['attack_cat'].map(attack_cat_mapping).fillna(0).astype(int)

print("✅ Categorical variables encoded successfully!")
df.head()

✅ Categorical variables encoded successfully!


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,0.121478,113,0,1,6,4,258,172,74.087490,252,...,1,1,0,0,0,1,1,0,0,0
1,0.649902,113,0,1,14,38,734,42014,78.473372,62,...,1,2,0,0,0,1,6,0,0,0
2,1.623129,113,0,1,8,16,364,13186,14.170161,62,...,1,3,0,0,0,2,6,0,0,0
3,1.681642,113,1,1,12,12,628,770,13.677108,62,...,1,3,1,1,0,2,1,0,0,0
4,0.449454,113,0,1,10,6,534,268,33.373826,254,...,1,40,0,0,0,2,39,0,0,0


#### Preparing X and Y variables

In [11]:
# For binary classification (Normal vs Attack)
y_binary = df['label']

# For multi-class classification (Attack types)
y_multiclass = df['attack_cat']

# Features (excluding target variables)
X = df.drop(columns=['label', 'attack_cat'])

print("Feature matrix shape:", X.shape)
print("Binary target shape:", y_binary.shape)
print("Multiclass target shape:", y_multiclass.shape)

# Display feature information
print("\nFeature types:")
print(X.dtypes.value_counts())

Feature matrix shape: (257673, 42)
Binary target shape: (257673,)
Multiclass target shape: (257673,)

Feature types:
int64      31
float64    11
Name: count, dtype: int64


#### Create Column Transformer for preprocessing

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# Identify numerical features
num_features = X.select_dtypes(include=[np.number]).columns.tolist()

print("Numerical features:", len(num_features))
print("All features are numerical - no need for one-hot encoding")

# Create preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), num_features)
    ]
)

print("✅ Preprocessor created successfully!")

Numerical features: 42
All features are numerical - no need for one-hot encoding
✅ Preprocessor created successfully!


#### Split dataset into train and test

In [13]:
from sklearn.model_selection import train_test_split

# For binary classification
X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

# For multiclass classification  
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X, y_multiclass, test_size=0.2, random_state=42, stratify=y_multiclass
)

print("Binary Classification Split:")
print(f"Training set: {X_train_bin.shape}, Test set: {X_test_bin.shape}")

print("\nMulticlass Classification Split:")
print(f"Training set: {X_train_multi.shape}, Test set: {X_test_multi.shape}")

# Apply preprocessing
X_train_bin_scaled = preprocessor.fit_transform(X_train_bin)
X_test_bin_scaled = preprocessor.transform(X_test_bin)

X_train_multi_scaled = preprocessor.fit_transform(X_train_multi)
X_test_multi_scaled = preprocessor.transform(X_test_multi)

print("✅ Data preprocessing completed!")

Binary Classification Split:
Training set: (206138, 42), Test set: (51535, 42)

Multiclass Classification Split:
Training set: (206138, 42), Test set: (51535, 42)
✅ Data preprocessing completed!


#### Create an Evaluate Function to give all metrics after model Training

In [14]:
def evaluate_model(true, predicted, model_name="", problem_type="binary"):
    """
    Evaluate model performance with multiple metrics
    """
    accuracy = accuracy_score(true, predicted)
    f1 = f1_score(true, predicted, average='weighted' if problem_type == "multiclass" else 'binary')
    
    if problem_type == "binary":
        auc = roc_auc_score(true, predicted)
    else:
        auc = roc_auc_score(true, predicted, multi_class='ovr', average='weighted')
    
    # Classification report
    report = classification_report(true, predicted)
    
    return {
        'accuracy': accuracy,
        'f1_score': f1,
        'auc_roc': auc,
        'report': report
    }

def train_and_evaluate_models(X_train, X_test, y_train, y_test, problem_type="binary"):
    """
    Train multiple models and evaluate their performance
    """
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "K-Neighbors": KNeighborsClassifier(),
        "Gaussian NB": GaussianNB(),
        "SVM": SVC(random_state=42),
        "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
        "CatBoost": CatBoostClassifier(verbose=False, random_state=42),
        "LightGBM": LGBMClassifier(random_state=42),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(random_state=42)
    }
    
    results = {}
    
    for name, model in models.items():
        print(f"\n🚀 Training {name}...")
        
        try:
            # Fit model
            if name in ["Logistic Regression", "SVM", "K-Neighbors", "Gaussian NB"]:
                # Use scaled data for these models
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
            else:
                # Use original data for tree-based models
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
            
            # Evaluate
            metrics = evaluate_model(y_test, y_pred, name, problem_type)
            results[name] = metrics
            
            print(f"✅ {name} - Accuracy: {metrics['accuracy']:.4f}, F1-Score: {metrics['f1_score']:.4f}")
            
        except Exception as e:
            print(f"❌ Error with {name}: {str(e)}")
            results[name] = None
    
    return results, models

### Binary Classification - Model Training

In [ ]:
print("🔍 BINARY CLASSIFICATION - NORMAL vs ATTACK")

# Train models with scaled data
results_binary, models_binary = train_and_evaluate_models(
    X_train_bin_scaled, X_test_bin_scaled, y_train_bin, y_test_bin, "binary"
)

🔍 BINARY CLASSIFICATION - NORMAL vs ATTACK

🚀 Training Logistic Regression...
✅ Logistic Regression - Accuracy: 0.9020, F1-Score: 0.9266

🚀 Training Random Forest...
✅ Random Forest - Accuracy: 0.9526, F1-Score: 0.9628

🚀 Training Decision Tree...
✅ Decision Tree - Accuracy: 0.9390, F1-Score: 0.9522

🚀 Training K-Neighbors...
✅ K-Neighbors - Accuracy: 0.9165, F1-Score: 0.9345

🚀 Training Gaussian NB...
✅ Gaussian NB - Accuracy: 0.8247, F1-Score: 0.8630

🚀 Training SVM...
✅ SVM - Accuracy: 0.9184, F1-Score: 0.9381

🚀 Training XGBoost...
✅ XGBoost - Accuracy: 0.9493, F1-Score: 0.9601

🚀 Training CatBoost...
✅ CatBoost - Accuracy: 0.9514, F1-Score: 0.9618

🚀 Training LightGBM...
[LightGBM] [Info] Number of positive: 131738, number of negative: 74400
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.026296 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info]

### Multiclass Classification - Model Training

In [ ]:
print("🔍 MULTICLASS CLASSIFICATION - ATTACK TYPES")

# Train models with scaled data
results_multi, models_multi = train_and_evaluate_models(
    X_train_multi_scaled, X_test_multi_scaled, y_train_multi, y_test_multi, "multiclass"
)

#### Display Results Comparison

In [ ]:
def display_results_comparison(results, title):
    """Display results in a comparative table"""
    print(f"\n{'='*80}")
    print(f"📊 {title} - MODEL PERFORMANCE COMPARISON")
    print(f"{'='*80}")
    
    # Create results dataframe
    results_df = pd.DataFrame({
        'Model': [],
        'Accuracy': [],
        'F1-Score': [],
        'AUC-ROC': []
    })
    
    for model_name, metrics in results.items():
        if metrics is not None:
            results_df = pd.concat([results_df, pd.DataFrame({
                'Model': [model_name],
                'Accuracy': [metrics['accuracy']],
                'F1-Score': [metrics['f1_score']],
                'AUC-ROC': [metrics['auc_roc']]
            })], ignore_index=True)
    
    # Sort by accuracy
    results_df = results_df.sort_values('Accuracy', ascending=False)
    
    # Display table
    print(results_df.to_string(index=False))
    
    return results_df

# Display binary results
binary_results_df = display_results_comparison(results_binary, "BINARY CLASSIFICATION")

# Display multiclass results  
multi_results_df = display_results_comparison(results_multi, "MULTICLASS CLASSIFICATION")

#### Visualize Model Performance


In [ ]:
def plot_model_comparison(results_df, title):
    """Plot model performance comparison"""
    plt.figure(figsize=(15, 8))
    
    # Set positions
    x = np.arange(len(results_df))
    width = 0.25
    
    # Create bars
    plt.bar(x - width, results_df['Accuracy'], width, label='Accuracy', alpha=0.8)
    plt.bar(x, results_df['F1-Score'], width, label='F1-Score', alpha=0.8)
    plt.bar(x + width, results_df['AUC-ROC'], width, label='AUC-ROC', alpha=0.8)
    
    # Customize plot
    plt.xlabel('Models')
    plt.ylabel('Scores')
    plt.title(f'Model Performance Comparison - {title}')
    plt.xticks(x, results_df['Model'], rotation=45, ha='right')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Plot comparisons
plot_model_comparison(binary_results_df, "Binary Classification")
plot_model_comparison(multi_results_df, "Multiclass Classification")

#### Confusion Matrix for Best Models


In [ ]:
def plot_confusion_matrices(best_binary_model, best_multi_model, models_dict):
    """Plot confusion matrices for best performing models"""
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    # Binary Classification Confusion Matrix
    if best_binary_model in models_dict:
        model = models_dict[best_binary_model]
        if hasattr(model, 'predict'):
            y_pred_bin = model.predict(X_test_bin_scaled)
            cm_bin = confusion_matrix(y_test_bin, y_pred_bin)
            
            sns.heatmap(cm_bin, annot=True, fmt='d', cmap='Blues', ax=axes[0])
            axes[0].set_title(f'Confusion Matrix - {best_binary_model}\n(Binary Classification)')
            axes[0].set_xlabel('Predicted')
            axes[0].set_ylabel('Actual')
    
    # Multiclass Classification Confusion Matrix
    if best_multi_model in models_dict:
        model = models_dict[best_multi_model]
        if hasattr(model, 'predict'):
            y_pred_multi = model.predict(X_test_multi_scaled)
            cm_multi = confusion_matrix(y_test_multi, y_pred_multi)
            
            sns.heatmap(cm_multi, annot=True, fmt='d', cmap='Blues', ax=axes[1])
            axes[1].set_title(f'Confusion Matrix - {best_multi_model}\n(Multiclass Classification)')
            axes[1].set_xlabel('Predicted')
            axes[1].set_ylabel('Actual')
    
    plt.tight_layout()
    plt.show()

# Get best models
best_binary_model = binary_results_df.iloc[0]['Model']
best_multi_model = multi_results_df.iloc[0]['Model']

print(f"🏆 Best Binary Classification Model: {best_binary_model}")
print(f"🏆 Best Multiclass Classification Model: {best_multi_model}")

plot_confusion_matrices(best_binary_model, best_multi_model, models_binary)

#### Feature Importance Analysis

In [ ]:
def plot_feature_importance(model, feature_names, top_n=15, title="Feature Importance"):
    """Plot feature importance for tree-based models"""
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        indices = np.argsort(importances)[::-1]
        
        plt.figure(figsize=(12, 8))
        plt.title(title)
        plt.bar(range(min(top_n, len(importances))), 
                importances[indices[:top_n]])
        plt.xticks(range(min(top_n, len(importances))), 
                  [feature_names[i] for i in indices[:top_n]], rotation=45)
        plt.tight_layout()
        plt.show()
        
        # Print top features
        print(f"Top {top_n} Most Important Features:")
        for i in range(min(top_n, len(importances))):
            print(f"{i+1}. {feature_names[indices[i]]}: {importances[indices[i]]:.4f}")
    else:
        print("Model doesn't have feature_importances_ attribute")

# Plot feature importance for best model
if best_binary_model in models_binary:
    best_model = models_binary[best_binary_model]
    plot_feature_importance(best_model, X.columns.tolist(), 
                          title=f"Feature Importance - {best_binary_model}")

#### Hyperparameter Tuning for Best Model


In [ ]:
def hyperparameter_tuning_best_model(model, param_grid, X_train, y_train, X_test, y_test):
    """Perform hyperparameter tuning for the best model"""
    print(f"🔧 Performing Hyperparameter Tuning for {model.__class__.__name__}...")
    
    # Use RandomizedSearchCV for faster tuning
    random_search = RandomizedSearchCV(
        model, param_grid, n_iter=20, cv=3, scoring='accuracy', 
        n_jobs=-1, random_state=42, verbose=1
    )
    
    random_search.fit(X_train, y_train)
    
    print(f"✅ Best parameters: {random_search.best_params_}")
    print(f"✅ Best cross-validation score: {random_search.best_score_:.4f}")
    
    # Evaluate on test set
    best_model = random_search.best_estimator_
    y_pred = best_model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_pred)
    
    print(f"✅ Test set accuracy: {test_accuracy:.4f}")
    
    return best_model

# Example: Tune Random Forest if it's the best model
if best_binary_model == "Random Forest":
    rf_param_grid = {
        'n_estimators': [50, 100, 200, 300],
        'max_depth': [10, 20, 30, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'bootstrap': [True, False]
    }
    
    tuned_rf = hyperparameter_tuning_best_model(
        models_binary["Random Forest"], rf_param_grid,
        X_train_bin, y_train_bin, X_test_bin, y_test_bin
    )

#### Cross-Validation Performance


In [ ]:
def perform_cross_validation(models_dict, X, y, cv=5):
    """Perform cross-validation for all models"""
    cv_results = {}
    
    for name, model in models_dict.items():
        print(f"🔍 Performing {cv}-fold CV for {name}...")
        
        try:
            # Use scaled data for certain models
            if name in ["Logistic Regression", "SVM", "K-Neighbors", "Gaussian NB"]:
                X_scaled = preprocessor.fit_transform(X)
                cv_scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='accuracy')
            else:
                cv_scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
            
            cv_results[name] = {
                'mean_score': cv_scores.mean(),
                'std_score': cv_scores.std(),
                'all_scores': cv_scores
            }
            
            print(f"✅ {name} - Mean CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
            
        except Exception as e:
            print(f"❌ CV failed for {name}: {str(e)}")
            cv_results[name] = None
    
    return cv_results

# Perform cross-validation for binary classification
print("🔍 CROSS-VALIDATION RESULTS - BINARY CLASSIFICATION")
cv_results_binary = perform_cross_validation(models_binary, X, y_binary)

# Perform cross-validation for multiclass classification
print("\n🔍 CROSS-VALIDATION RESULTS - MULTICLASS CLASSIFICATION")
cv_results_multi = perform_cross_validation(models_multi, X, y_multiclass)

#### Model Persistence - Save Best Models


In [ ]:
import joblib
import pickle

def save_models(best_binary_model, best_multi_model, models_dict, preprocessor):
    """Save the best models and preprocessor for deployment"""
    
    # Create models directory
    import os
    os.makedirs('saved_models', exist_ok=True)
    
    # Save best binary model
    if best_binary_model in models_dict:
        binary_model = models_dict[best_binary_model]
        joblib.dump(binary_model, f'saved_models/best_binary_model.pkl')
        print(f"✅ Saved best binary model: {best_binary_model}")
    
    # Save best multiclass model
    if best_multi_model in models_dict:
        multi_model = models_dict[best_multi_model]
        joblib.dump(multi_model, f'saved_models/best_multiclass_model.pkl')
        print(f"✅ Saved best multiclass model: {best_multi_model}")
    
    # Save preprocessor
    joblib.dump(preprocessor, 'saved_models/preprocessor.pkl')
    print("✅ Saved preprocessor")
    
    # Save label encoders
    encoders = {
        'proto_encoder': le_proto,
        'service_mapping': service_mapping,
        'state_mapping': state_mapping,
        'attack_cat_mapping': attack_cat_mapping
    }
    
    with open('saved_models/encoders.pkl', 'wb') as f:
        pickle.dump(encoders, f)
    print("✅ Saved label encoders")

# Save models
save_models(best_binary_model, best_multi_model, models_binary, preprocessor)

#### Final Performance Summary


In [ ]:
print("🎯 FINAL MODEL PERFORMANCE SUMMARY")

print(f"\n🏆 BEST BINARY CLASSIFICATION MODEL: {best_binary_model}")
best_binary_metrics = results_binary[best_binary_model]
print(f"   Accuracy: {best_binary_metrics['accuracy']:.4f}")
print(f"   F1-Score: {best_binary_metrics['f1_score']:.4f}")
print(f"   AUC-ROC:  {best_binary_metrics['auc_roc']:.4f}")

print(f"\n🏆 BEST MULTICLASS CLASSIFICATION MODEL: {best_multi_model}")
best_multi_metrics = results_multi[best_multi_model]
print(f"   Accuracy: {best_multi_metrics['accuracy']:.4f}")
print(f"   F1-Score: {best_multi_metrics['f1_score']:.4f}")
print(f"   AUC-ROC:  {best_multi_metrics['auc_roc']:.4f}")

print(f"\n📊 DATASET STATISTICS:")
print(f"   Total samples: {len(df)}")
print(f"   Normal traffic: {len(df[df['label'] == 0])} ({len(df[df['label'] == 0])/len(df)*100:.2f}%)")
print(f"   Attack traffic: {len(df[df['label'] == 1])} ({len(df[df['label'] == 1])/len(df)*100:.2f}%)")

print(f"\n🔧 FEATURES:")
print(f"   Total features: {X.shape[1]}")
print(f"   Numerical features: {len(num_features)}")

print(f"\n💾 MODELS SAVED:")
print("   ✅ Best binary classification model")
print("   ✅ Best multiclass classification model") 
print("   ✅ Preprocessor")
print("   ✅ Label encoders")

#### Model Deployment Ready Function


In [ ]:
def predict_network_traffic(features, model_type='binary'):
    """
    Function to predict network traffic (ready for deployment)
    
    Parameters:
    features: array-like, shape (n_samples, n_features)
    model_type: 'binary' or 'multiclass'
    
    Returns:
    predictions: array, shape (n_samples,)
    """
    try:
        # Load preprocessor and model
        preprocessor = joblib.load('saved_models/preprocessor.pkl')
        
        if model_type == 'binary':
            model = joblib.load('saved_models/best_binary_model.pkl')
        else:
            model = joblib.load('saved_models/best_multiclass_model.pkl')
        
        # Preprocess features
        features_scaled = preprocessor.transform(features)
        
        # Make predictions
        predictions = model.predict(features_scaled)
        
        return predictions
        
    except Exception as e:
        print(f"❌ Prediction error: {str(e)}")
        return None

print("✅ Model training completed successfully!")
print("🚀 Models are ready for deployment!")